In [ ]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

In [ ]:
import os
if not os.path.exists('/kaggle/working/F5-TTS'):
    !git clone https://github.com/SWivid/F5-TTS.git /kaggle/working/F5-TTS
else:
    print('F5-TTS already cloned, skipping.')


In [ ]:
%cd /kaggle/working/F5-TTS

In [ ]:
!pip install -e .

In [ ]:
# Sanity check: confirm F5-TTS actually installed before doing anything else
!pip show f5-tts
!python -m f5_tts.infer.infer_cli --help


## Serve F5-TTS as an API for the Steamroller app
Run the three cells below (after the setup cells above have already run once) to expose this notebook as an HTTP endpoint your local Streamlit app can call.

In [ ]:
!pip install -q flask pyngrok


In [ ]:
import os
import sys
import uuid
import subprocess
from flask import Flask, request, send_file

app = Flask(__name__)

WORKDIR = "/kaggle/working"
UPLOAD_DIR = os.path.join(WORKDIR, "uploads")
OUTPUT_DIR = os.path.join(WORKDIR, "output")
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)


@app.route("/health", methods=["GET"])
def health():
    return {"status": "ok"}


@app.route("/clone", methods=["POST"])
def clone():
    text = request.form.get("text")
    ref_text = request.form.get("ref_text", "")
    speaker_file = request.files.get("speaker_wav")

    if not text or not speaker_file:
        return {"error": "text and speaker_wav are required"}, 400

    uid = uuid.uuid4().hex
    ref_path = os.path.join(UPLOAD_DIR, f"ref_{uid}.wav")
    speaker_file.save(ref_path)

    output_filename = f"clone_{uid}.wav"

    cmd = [
        sys.executable, "-m", "f5_tts.infer.infer_cli",
        "--model", "F5TTS_Base",
        "--ref_audio", ref_path,
        "--ref_text", ref_text,
        "--gen_text", text,
        "--output_dir", OUTPUT_DIR,
        "--output_file", output_filename,
        "--remove_silence",
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    if result.returncode != 0:
        return {"error": result.stderr[-2000:]}, 500

    output_path = os.path.join(OUTPUT_DIR, output_filename)
    if not os.path.exists(output_path):
        return {"error": "F5-TTS ran but no output file was produced", "log": result.stdout[-2000:]}, 500

    return send_file(output_path, mimetype="audio/wav")


In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token("your_token")  # get a free token at https://ngrok.com
public_url = ngrok.connect(5000)
print("Public URL:", public_url)
print("Copy this into the KAGGLE_TTS_URL environment variable on your local machine.")

app.run(port=5000)
